# Lesson 12 - Сокращение истории чата с помощью агентской записной книжки

В этой тетради показано, как управлять контекстом в длинных разговорах с использованием Microsoft Agent Framework. По мере роста диалогов количество токенов увеличивается — в конечном итоге превышая окно контекста модели. Мы решаем эту проблему с помощью **паттерна суммирования контекста** и **агентской записной книжки** для постоянной памяти.

## Чему вы научитесь:
1. **Почему важно управление контекстом**: понимание лимитов токенов и окон контекста  
2. **Агенты, учитывающие контекст**: создание агентов, которые управляют собственным контекстом разговора  
3. **Паттерн суммирования контекста**: использование инструментов для сжатия истории разговора  
4. **Агентская записная книжка**: постоянная память, которая сохраняется при сокращении контекста  

## Необходимые знания:
- Настройка Azure OpenAI с конфигурированными переменными окружения  
- Понимание базовых понятий агентов из предыдущих уроков


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Почему важно управление контекстом

У каждого LLM есть ограниченное **окно контекста** — максимальное количество токенов, которые он может обработать за один запрос. В ходе многошагового диалога:

- **Количество токенов растет линейно** с каждым сообщением пользователя и ответом ассистента.
- **Токены подсказки доминируют в стоимости**, поскольку вся история отправляется заново на каждом шаге.
- В конечном итоге диалог **превышает окно контекста**, и модель либо усекает текст, либо возвращает ошибку.

### Стратегии управления контекстом

| Стратегия | Как работает | Компромисс |
|---|---|---|
| **Усечение** | Удаление самых старых сообщений | Потеря раннего контекста |
| **Резюме** | Сжатие старых сообщений в сводку | Часть деталей теряется, но сохраняются ключевые моменты |
| **Заметки / Внешняя память** | Хранение ключевых фактов вне диалога | Требуются вызовы инструментов, но сохраняется при любом сокращении |

В этой тетрадке мы комбинируем **резюме** с инструментом **заметок**, чтобы агент мог поддерживать непрерывность даже при сокращении истории диалога.


## Создание контекстно-зависимого агента


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Симуляция долгого разговора

Давайте пройдем через многошаговый разговор, чтобы увидеть, как накапливается контекст. Агент должен сохранять ключевые детали (предпочтения, бюджет, даты путешествия) на протяжении всех этапов и демонстрировать непрерывность.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обратите внимание, как агент сохраняет контекст из предыдущих ходов — он знает о Японии, суши, храмах, фотографии, бюджете в 3000 долларов, путешествии в одиночку и периоде в апреле. В короткой беседе это работает хорошо, но по мере роста беседы отправлять всю историю становится дорого.

Давайте продолжим разговор с большим количеством ходов, чтобы увидеть накопление контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Паттерн Суммирования Контекста

По мере роста разговора мы можем использовать **инструмент суммирования**, чтобы сжать накопленный контекст в компактный формат. Агент вызывает этот инструмент для записи ключевых предпочтений, чтобы даже если старые сообщения будут удалены, важная информация сохранилась.

Этот паттерн является строительным блоком для более сложного сокращения истории:
1. Агент определяет ключевые факты из разговора
2. Он вызывает инструмент суммирования для их сохранения
3. Старые сообщения могут быть безопасно удалены, поскольку сводка фиксирует важное

Ниже мы определяем инструмент `summarize_preferences`, который агент может вызвать для записи компактного резюме того, что он узнал.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резюме

В этом уроке вы узнали, как управлять контекстом в длительных разговорах с агентом, используя Microsoft Agent Framework:

### Ключевые понятия
- **Окна контекста конечны** — каждый токен в истории разговора стоит денег и учитывается в лимит.
- **Инструменты суммаризации** позволяют агенту сжимать накопленный контекст в компактные сводки, снижая использование токенов при сохранении важной информации.
- **Блокноты агента** предоставляют постоянную внешнюю память, которая сохраняется при любом сокращении разговора.

### Что вы создали
- **Агент с осознанием контекста**, который поддерживает непрерывность в многоходовых разговорах
- **Инструмент суммаризации** (`summarize_preferences`), который записывает ключевые данные пользователя в компактном формате
- **Многоходовой разговор**, демонстрирующий сохранение контекста и обработку изменений

### Практические применения
- **Боты поддержки клиентов**: запоминают предпочтения на протяжении длительных сессий поддержки
- **Персональные помощники**: отслеживают текущие проекты без повторного объяснения контекста
- **Образовательные репетиторы**: сохраняют прогресс ученика на протяжении многих взаимодействий

### Следующие шаги
- Реализовать полноценный инструмент блокнота с файловым хранением
- Добавить автоматическое усечение истории после суммаризации
- Совместить с векторными базами данных для семантического поиска памяти
- Создать агентов, которые могут возобновлять разговоры спустя дни с полным контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведён с помощью сервиса автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, просим учитывать, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на исходном языке следует считать достоверным источником информации. Для важной или критичной информации рекомендуется использовать профессиональный перевод, выполненный человеком. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
